In [1]:
import numpy as np
import matplotlib.pyplot as plt
import random
import time
from IPython.display import display, clear_output
import ipywidgets as widgets

class SchellingModel:
    """
    DEFINITIONS:
    1. Agents: Individuals belonging to different groups (e.g., red and blue).

    2. Grid: A 2D lattice (e.g., 3×3, 10×10).

    3.Empty cells: Vacant houses.

    4. Neighborhood: Typically the 8 surrounding cells (Moore neighborhood).

    5. THRESHOLD (tau): The 'Tolerance Level'. The minimum percentage of neighbors
       an agent requires to be of the same type to stay in their current location.
       Formula: Satisfaction = (Same Neighbors) / (Total Occupied Neighbors)

    6. DENSITY: The percentage of the grid occupied by agents.
       Formula: Total Agents = (Grid Size^2) * Density

    7. MINORITY RATIO: The balance between the two groups (e.g., 0.5 is an equal split).
    """

    def __init__(self, size=20, density=0.8, threshold=0.4, minority_ratio=0.5):
        self.size = size
        self.threshold = threshold
        self.density = density
        self.minority_ratio = minority_ratio
        self.history_satisfaction = []
        self.reset()

    def reset(self):
        """
        STEP 1: INITIALIZATION
        - Create an empty grid (0 = Empty).
        - Calculate population based on Density and Minority Ratio.
        - Randomly distribute agents across the grid.
        """
        self.grid = np.zeros((self.size, self.size))
        total_cells = self.size * self.size
        num_agents = int(total_cells * self.density)

        # Identify all coordinates and shuffle them to ensure random placement
        all_pos = [(r, c) for r in range(self.size) for c in range(self.size)]
        occupied_pos = random.sample(all_pos, num_agents)

        # Divide the population into Group 1 (Blue) and Group 2 (Red)
        num_group_a = int(num_agents * self.minority_ratio)
        for i, (r, c) in enumerate(occupied_pos):
            self.grid[r, c] = 1 if i < num_group_a else 2

        # Initialize history with the starting satisfaction level
        self.history_satisfaction = [self.calculate_mean_sat()]

    def get_neighbors_info(self, r, c):
        """
        STEP 2: LOCAL SATISFACTION CALCULATION
        - Logic: Uses the Moore Neighborhood (the 8 surrounding cells).
        - Formula: Similarity Ratio (S) = N_similar / N_occupied_neighbors
        - Condition: If S < Threshold, the agent is 'Unhappy'.
        """
        agent_type = self.grid[r, c]
        similar, total_occupied = 0, 0

        # Iterate through the 3x3 area around the agent
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0: continue # Skip the agent's own cell

                nr, nc = r + dr, c + dc
                # Check grid boundaries
                if 0 <= nr < self.size and 0 <= nc < self.size:
                    neighbor = self.grid[nr, nc]
                    if neighbor != 0: # Only occupied cells contribute to social pressure
                        total_occupied += 1
                        if neighbor == agent_type:
                            similar += 1

        # If an agent has no neighbors, they are satisfied by default (S=1.0)
        if total_occupied == 0: return 1.0, True

        sat = similar / total_occupied
        return sat, sat >= self.threshold

    def calculate_mean_sat(self):
        """
        METRIC: GLOBAL MEAN SATISFACTION
        Calculates the average similarity ratio across the entire population.
        As segregation increases, this number rises.
        """
        sats = []
        for r in range(self.size):
            for c in range(self.size):
                if self.grid[r, c] != 0:
                    sat, _ = self.get_neighbors_info(r, c)
                    sats.append(sat)
        return np.mean(sats) if sats else 0

    def step(self):
        """
        STEP 3: MOVEMENT LOGIC
        - Identify all 'Unhappy' agents.
        - Identify all 'Empty' cells.
        - Relocate unhappy agents to random empty spots.
        """
        unhappy, empty = [], []
        for r in range(self.size):
            for c in range(self.size):
                if self.grid[r, c] == 0:
                    empty.append((r, c))
                else:
                    _, happy = self.get_neighbors_info(r, c)
                    if not happy:
                        unhappy.append((r, c))

        # Move agents
        random.shuffle(unhappy)
        for r, c in unhappy:
            if not empty: break
            # Select a random destination from vacant spots
            dest_idx = random.randint(0, len(empty) - 1)
            new_r, new_c = empty.pop(dest_idx)

            # Transfer agent and free up old cell
            self.grid[new_r, new_c] = self.grid[r, c]
            self.grid[r, c] = 0
            empty.append((r, c)) # The old spot is now empty

        # Record the new global satisfaction for the graph
        self.history_satisfaction.append(self.calculate_mean_sat())
        return len(unhappy)

# --- VISUALIZATION ENGINE ---

model = SchellingModel()
output = widgets.Output()

def visualize():
    """
    Renders the current state of the grid using circular nodes
    and updates the satisfaction trend line.
    """
    with output:
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

        # Plot 1: The Network (Scatter plot creates the circular node look)
        r1, c1 = np.where(model.grid == 1)
        r2, c2 = np.where(model.grid == 2)
        ax1.scatter(c1, r1, color='#3498db', s=120, edgecolors='white', label='Group A')
        ax1.scatter(c2, r2, color='#e74c3c', s=120, edgecolors='white', label='Group B')

        ax1.set_title(f"Schelling Grid (Threshold: {model.threshold*100:.0f}%)")
        ax1.set_facecolor('#f4f4f4')
        ax1.set_xticks([]); ax1.set_yticks([])
        ax1.set_xlim(-1, model.size); ax1.set_ylim(-1, model.size)
        ax1.invert_yaxis()

        # Plot 2: Satisfaction Over Time (The Segregation Metric)
        ax2.plot(model.history_satisfaction, color='#27ae60', marker='o', linewidth=2)
        ax2.set_title("Global Mean Satisfaction (Metric)")
        ax2.set_xlabel("Time Step (Iteration)"); ax2.set_ylabel("Similarity Ratio")
        ax2.set_ylim(0, 1.05); ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

# --- INTERACTIVE CONTROLS (IPYWIDGETS) ---

s_thresh = widgets.FloatSlider(value=0.4, min=0, max=1.0, step=0.05, description='Threshold')
s_dens = widgets.FloatSlider(value=0.8, min=0.1, max=0.95, step=0.05, description='Density')
s_ratio = widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description='Minority %')

btn_reset = widgets.Button(description="Reset Grid", button_style='warning')
btn_step = widgets.Button(description="Single Step", button_style='info')
btn_run = widgets.Button(description="Run Full Simulation", button_style='success')

def on_reset(b):
    model.threshold = s_thresh.value
    model.density = s_dens.value
    model.minority_ratio = s_ratio.value
    model.reset()
    visualize()

def on_step(b):
    model.step()
    visualize()

def on_run(b):
    for i in range(50): # Cap at 50 steps to prevent infinite loops
        moves = model.step()
        visualize()
        if moves == 0: break # System has reached equilibrium
        time.sleep(0.05)

# Mapping click events to functions
btn_reset.on_click(on_reset)
btn_step.on_click(on_step)
btn_run.on_click(on_run)

# Layout: UI Sliders at top, Buttons in middle, Graph at bottom
display(widgets.VBox([
    widgets.Label(value="Adjust Simulation Parameters:"),
    widgets.HBox([s_thresh, s_dens, s_ratio]),
    widgets.HBox([btn_reset, btn_step, btn_run]),
    output
]))

# Initial render
on_reset(None)